# 📄 LLaMA Fine-Tuning — Legal Document Summarization PoC
**Author:** Sanusi Isiaka Olatunji | M.Sc. Data Science | University of Leoben
**Context:** Recreates the AIM professional project — fine-tuning a LLaMA-based model for legal document summarization

---

### ⚠️ Important — read this first
**Runtime → Change runtime type → GPU (T4)** — fine-tuning needs a GPU, even a small one.

### Experiment design — intentional constraints

This PoC deliberately uses a **small open-source model** (TinyLlama-1.1B, same
architecture family as LLaMA) instead of full LLaMA-7B, for one reason:
**it runs entirely free on Colab's GPU**, with no gated model access and no
paid compute required.

This is a known and intentional tradeoff:

| | Production setup (AIM, professional) | This PoC (Colab, free) |
|---|---|---|
| Base model | LLaMA-7B+ | TinyLlama-1.1B |
| Training data | Hundreds–thousands of examples | 12 examples |
| Compute | Dedicated GPU infrastructure | Free Colab T4 |
| Goal | Production-quality summarization | Demonstrate the full pipeline mechanics |

**Expected outcome:** at this reduced scale, the model is **capacity-constrained**
— it may not produce production-quality summaries. The goal of this notebook
is to demonstrate the complete fine-tuning pipeline (data prep → LoRA
configuration → training → evaluation → systematic debugging) and to
clearly document where small-model capacity limits emerge — a real and
common finding in applied LLM fine-tuning work.

### What makes THIS project different from the other two:

| | AT&S Project | AVL Project | **This Project (LLaMA)** |
|---|---|---|---|
| **What it does** | Chat with a pre-built AI | Search documents + chat | **Actually retrains the AI's brain** |
| **Model used** | Gemini (via API, untouched) | Gemini (via API, untouched) | **LLaMA (downloaded, modified)** |
| **Where it runs** | Google's servers | Google's servers | **Your own GPU** |
| **What changes** | Nothing — same model every time | Nothing — same model every time | **The model's internal weights change** |
| **Analogy** | Asking a librarian a question | Asking a librarian to fetch + explain a book | **Sending an employee to a training course** |

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers datasets peft accelerate trl matplotlib pandas
print('✅ Dependencies installed (no bitsandbytes needed — using full precision)')
print('   transformers — loads LLaMA-architecture model')
print('   peft         — LoRA fine-tuning (lightweight, fits on free GPU)')
print('   trl          — training loop for language models')

In [ ]:
# ── CELL 2: Configuration ─────────────────────────────────────────────────────
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # open, free, LLaMA-architecture model
OUTPUT_DIR = "./legal-summarizer-lora"

print(f'✅ Configuration set')
print(f'   Base model : {MODEL_NAME}')
print(f'   GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️ No GPU detected — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── CELL 3: Build legal document summarization dataset ───────────────────────
legal_training_examples = [
    {"document": "This Non-Disclosure Agreement (NDA) is entered into between Party A and Party B effective as of the date of signing. Party B agrees to maintain in confidence all proprietary information disclosed by Party A, including but not limited to technical data, business plans, and customer lists. This obligation shall survive for a period of five (5) years following termination of this Agreement. Party B shall not disclose such information to any third party without prior written consent from Party A.",
     "summary": "NDA requiring Party B to keep Party A's confidential information private for 5 years after the agreement ends, with no disclosure to third parties without written consent."},
    {"document": "This Employment Agreement sets forth the terms under which the Employee shall provide services to the Employer. The Employee shall be entitled to an annual salary of the agreed amount, payable in monthly installments. The Employee may be terminated for cause, including but not limited to gross misconduct, breach of company policy, or failure to perform assigned duties. Upon termination, the Employee shall return all company property within five business days.",
     "summary": "Employment agreement covering salary terms, grounds for termination (misconduct, policy breach, poor performance), and a 5-day window to return company property after termination."},
    {"document": "This Lease Agreement grants the Tenant the right to occupy the Premises for a term of twelve (12) months commencing on the Effective Date. Monthly rent shall be due on the first day of each month. Late payments exceeding five (5) days shall incur a penalty fee of 5% of the monthly rent. The Tenant is responsible for all utility costs unless otherwise stated. Either party may terminate this Lease with sixty (60) days written notice.",
     "summary": "12-month lease with rent due monthly, a 5% late fee after 5 days, tenant-paid utilities, and a 60-day notice period required for either party to terminate."},
    {"document": "This Software License Agreement permits the Licensee to use the Software solely for internal business purposes. The Licensee shall not reverse engineer, decompile, or redistribute the Software without express written permission from the Licensor. The license is granted on a non-exclusive, non-transferable basis. The Licensor provides the Software 'as is' without warranty of any kind, and shall not be liable for any indirect or consequential damages arising from its use.",
     "summary": "Non-exclusive software license for internal use only, prohibiting reverse engineering or redistribution. Software provided 'as is' with no warranty and no liability for indirect damages."},
    {"document": "This Service Level Agreement defines the performance standards for services provided by the Vendor. The Vendor guarantees 99.5% uptime measured monthly. In the event of failure to meet this standard, the Vendor shall issue service credits equal to 5% of the monthly fee for each percentage point below the guaranteed uptime. Scheduled maintenance windows are excluded from uptime calculations, provided at least 48 hours notice is given.",
     "summary": "SLA guaranteeing 99.5% monthly uptime, with 5% service credits per percentage point shortfall. Scheduled maintenance (with 48-hour notice) doesn't count against uptime."},
    {"document": "This Purchase Agreement governs the sale of goods between Seller and Buyer. Title and risk of loss shall transfer to the Buyer upon delivery to the carrier. The Buyer shall inspect the goods within ten (10) business days of receipt and notify the Seller of any defects. Failure to provide timely notice shall constitute acceptance of the goods as delivered. All disputes arising under this Agreement shall be resolved through binding arbitration.",
     "summary": "Purchase agreement where risk transfers at delivery to carrier. Buyer has 10 business days to report defects or the goods are deemed accepted. Disputes go to binding arbitration."},
    {"document": "This Partnership Agreement establishes the terms of a general partnership between the Partners for the purpose of operating a joint business venture. Profits and losses shall be shared equally among the Partners unless otherwise agreed in writing. Each Partner shall have equal voting rights in major business decisions. Any Partner wishing to withdraw must provide ninety (90) days written notice to the remaining Partners.",
     "summary": "Partnership agreement with equal profit/loss sharing and equal voting rights, requiring 90 days written notice for any partner to withdraw."},
    {"document": "This Loan Agreement sets forth the terms under which the Lender shall provide a loan to the Borrower in the principal amount stated herein. The Borrower agrees to repay the loan with interest at a fixed annual rate over a term of five (5) years. Failure to make payment within fifteen (15) days of the due date shall constitute default, entitling the Lender to demand immediate repayment of the full outstanding balance.",
     "summary": "Loan agreement with fixed interest over a 5-year term. A payment more than 15 days late counts as default, allowing the Lender to demand full repayment immediately."},
    {"document": "This Consulting Agreement engages the Consultant to provide professional advisory services to the Client on an independent contractor basis. The Consultant shall not be considered an employee of the Client for any purpose, including tax withholding or benefits eligibility. The Consultant retains ownership of any pre-existing intellectual property but assigns to the Client all rights to deliverables created specifically under this engagement.",
     "summary": "Independent contractor consulting agreement. Consultant is not an employee (no tax withholding or benefits). Consultant keeps pre-existing IP but assigns new deliverables' rights to the Client."},
    {"document": "This Settlement Agreement resolves all claims between the Parties arising from the dispute described herein. In exchange for the payment specified, the Claimant releases the Respondent from any and all liability related to this matter. Both Parties agree to keep the terms of this Settlement confidential and shall not disclose the settlement amount to any third party except as required by law.",
     "summary": "Settlement resolving a dispute through a payment in exchange for full release of liability. Both parties must keep the settlement terms and amount confidential, except where legally required to disclose."},
    {"document": "This Distribution Agreement grants the Distributor the exclusive right to sell and distribute the Manufacturer's products within the defined Territory. The Distributor shall purchase a minimum quantity of units annually to maintain exclusivity. The Manufacturer reserves the right to terminate exclusivity if minimum purchase requirements are not met for two consecutive years.",
     "summary": "Exclusive distribution agreement requiring the Distributor to meet annual minimum purchase quantities. Manufacturer can revoke exclusivity after two consecutive years of missed minimums."},
    {"document": "This Intellectual Property Assignment Agreement transfers all rights, title, and interest in the specified inventions from the Assignor to the Assignee. The Assignor warrants that the inventions are original and do not infringe upon any third-party rights. The Assignee shall be solely responsible for filing and maintaining any related patent applications going forward.",
     "summary": "IP assignment transferring full ownership of inventions to the Assignee, with the Assignor warranting originality and non-infringement. Assignee handles all future patent filing and maintenance."},
]

print(f'✅ Training dataset created — {len(legal_training_examples)} document/summary pairs')
print(f'   (In production at AIM, this would be 100s–1000s of real legal documents)')
for i, ex in enumerate(legal_training_examples[:2], 1):
    print(f'\n  Example {i}:')
    print(f'  Document: {ex["document"][:100]}...')
    print(f'  Summary : {ex["summary"]}')

In [ ]:
# ── CELL 4: Load model — full precision, no quantization ─────────────────────
# NOTE: We use full precision (not 4-bit quantization) because bitsandbytes
# had unresolvable version conflicts in the Colab environment during testing.
# TinyLlama-1.1B is small enough to run in float16 on a free T4 GPU anyway.
print('🔄 Loading model (full precision, no quantization)...')

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f'✅ Model loaded: {MODEL_NAME}')
print(f'   Parameters: ~1.1 billion')
print(f'   Precision: float16 (no quantization)')

In [ ]:
# ── CELL 5: Sanity check — confirm base model loads and generates correctly ──
# This step is critical: it isolates infrastructure problems (broken libraries,
# corrupted weights) from actual fine-tuning problems, BEFORE we start training.
sanity_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Summarize this in one sentence: The sky is blue because of Rayleigh scattering."}],
    tokenize=False, add_generation_prompt=True
)
sanity_inputs = tokenizer(sanity_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    sanity_output = model.generate(
        **sanity_inputs, max_new_tokens=50, do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
result = tokenizer.decode(sanity_output[0], skip_special_tokens=True)
print('Sanity check output:')
print(result)
print()
if 'Rayleigh' in result or 'scattering' in result.lower() or 'blue' in result.lower():
    print('✅ Base model is healthy and generates coherent text — safe to proceed')
else:
    print('⚠️  Output looks incoherent — investigate before proceeding to fine-tuning')

In [ ]:
# ── CELL 6: Configure LoRA ─────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

print('✅ LoRA configured')
print(f'   Trainable parameters : {trainable_params:,} ({trainable_params/total_params*100:.3f}% of total)')
print(f'   Total model parameters: {total_params:,}')
print(f'   Target modules: all attention + MLP layers (wide adaptation)')

In [ ]:
# ── CELL 7: Format dataset using the model's OWN chat template ───────────────
from datasets import Dataset

def format_prompt(example):
    messages = [
        {"role": "system", "content": "You are a legal document summarization assistant. Summarize the given legal document concisely and accurately."},
        {"role": "user", "content": f"Summarize this legal document:\n\n{example['document']}"},
        {"role": "assistant", "content": example['summary']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

formatted_data = [format_prompt(ex) for ex in legal_training_examples]
train_dataset = Dataset.from_list(formatted_data)

print('✅ Dataset formatted using model\'s native chat template')
print(f'   Training examples: {len(train_dataset)}')
print(f'\n--- Sample formatted prompt ---')
print(train_dataset[0]['text'][:400] + '...')

In [ ]:
# ── CELL 8: Fine-tune the model ────────────────────────────────────────────────
print('🔄 Fine-tuning starting...\n')

from trl import SFTTrainer, SFTConfig
import time

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=8e-5,
    warmup_steps=3,
    max_grad_norm=0.3,
    logging_steps=2,
    save_strategy="no",
    bf16=True,
    fp16=False,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f'\n✅ Fine-tuning complete in {elapsed/60:.1f} minutes')
print(f'   Final training loss: {train_result.training_loss:.4f}')

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'\n✅ Fine-tuned model saved to {OUTPUT_DIR}')

In [ ]:
# ── CELL 9: Test the fine-tuned model on a NEW legal document ────────────────
test_document = """This Confidentiality and Non-Compete Agreement restricts the Employee from 
engaging in any business that directly competes with the Employer within a 50-mile radius 
for a period of two (2) years following termination of employment. The Employee further 
agrees not to solicit any clients or employees of the Employer during this restricted period. 
Violation of this Agreement entitles the Employer to seek injunctive relief and monetary damages."""

messages = [
    {"role": "system", "content": "You are a legal document summarization assistant. Summarize the given legal document concisely and accurately."},
    {"role": "user", "content": f"Summarize this legal document:\n\n{test_document}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.4,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output[0], skip_special_tokens=True)

print('📄 Test Document:')
print('─'*70)
print(test_document)
print('\n🤖 Fine-Tuned Model\'s Summary:')
print('─'*70)
input_decoded = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
print(generated[len(input_decoded):])

In [ ]:
# ── CELL 10: Visualise training progress ──────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

CB='#0d1117'; CP='#161b22'; CBR='#30363d'
CT='#e6edf3'; CM='#8b949e'; CN='#58a6ff'; CA='#f85149'; CG='#3fb950'

plt.rcParams.update({
    'figure.facecolor':CB,'axes.facecolor':CP,'axes.edgecolor':CBR,
    'axes.labelcolor':CT,'xtick.color':CM,'ytick.color':CM,
    'text.color':CT,'grid.color':CBR,'grid.linestyle':'--','grid.linewidth':0.5,
    'font.family':'monospace'
})

log_history = trainer.state.log_history
losses = [log['loss'] for log in log_history if 'loss' in log]
steps  = list(range(1, len(losses) + 1))

fig = plt.figure(figsize=(16, 6), facecolor=CB)
fig.suptitle(
    'LLaMA Fine-Tuning — Legal Document Summarization\n'
    'Sanusi Isiaka Olatunji  |  M.Sc. Data Science  |  University of Leoben',
    fontsize=12, color=CT, fontweight='bold', y=1.02
)
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, losses, color=CN, lw=2, marker='o', ms=5)
ax1.set_title('Training Loss Over Time', fontsize=10, color=CT)
ax1.set_xlabel('Training Step', fontsize=9)
ax1.set_ylabel('Loss (lower = better)', fontsize=9)
ax1.grid(True, alpha=0.4)

ax2 = fig.add_subplot(gs[0, 1])
ax2.axis('off')
summary_text = [
    f'Base model        : TinyLlama-1.1B-Chat',
    f'Architecture      : LLaMA family',
    f'Fine-tuning method: LoRA (r=16, all layers)',
    f'Training examples : {len(train_dataset)}',
    f'Epochs            : 5',
    f'Final loss        : {train_result.training_loss:.4f}',
    f'Training time     : {elapsed/60:.1f} min',
    f'Hardware          : Free Colab T4 GPU',
    f'Cost              : FREE ✅',
]
ax2.text(0.05, 0.95, '\n'.join(summary_text), transform=ax2.transAxes,
         fontsize=10, color=CT, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor=CP, edgecolor=CBR, alpha=0.9))
ax2.set_title('Fine-Tuning Summary', fontsize=10, color=CT)

plt.savefig('llama_finetune_results.png', dpi=150, bbox_inches='tight', facecolor=CB)
plt.show()
print('✅ Results chart saved → llama_finetune_results.png')

In [ ]:
# ── CELL 11: Findings & Conclusion ─────────────────────────────────────────────
print("="*70)
print("  PoC FINDINGS — Model Capacity vs Task Complexity")
print("="*70)
print("""
SUMMARY OF EXPERIMENTS:

This PoC tested LoRA fine-tuning of TinyLlama-1.1B-Chat for legal document
summarization across multiple configurations:

  Config                    | LoRA Rank | Target Modules     | LR     | Result
  ---------------------------|-----------|--------------------|--------|------------------
  Initial (baseline)         | r=8       | q,v only           | 2e-4   | Token collapse
  Conservative                | r=8       | q,v only           | 5e-5   | Generic/off-topic
  Wider dataset (12 ex.)     | r=8       | q,v only           | 1.5e-4 | Generic/off-topic
  Wide LoRA, high LR         | r=16      | all 7 modules      | 1e-4   | Token collapse
  Wide LoRA, low LR (final)  | r=16      | all 7 modules      | 8e-5   | Mixed/partial collapse

CONCLUSION:

Across all configurations, the model consistently struggled to produce
fully reliable, on-task legal summaries. Two failure modes alternated
depending on LoRA strength and learning rate:

  1. UNDERFITTING — model defaults to generic, unrelated text
  2. OVERFITTING / COLLAPSE — model degenerates into repeated tokens or
     unrelated patterns (code fragments, foreign tokens) drawn from its
     original pretraining data

ROOT CAUSE:

TinyLlama-1.1B-Chat has approximately 1.1 billion parameters — roughly 1/6th
the size of full LLaMA-7B. At this scale, with only 12 training examples,
the model lacks sufficient capacity to reliably override its general-purpose
pretrained behavior and consistently follow a narrow instruction-following
task like structured legal summarization.

This mirrors a known finding in LLM fine-tuning literature: smaller models
require either (a) significantly larger fine-tuning datasets (typically
hundreds to thousands of examples), (b) full fine-tuning rather than
parameter-efficient methods like LoRA, or (c) a larger base model with more
inherent capacity to specialize without losing general coherence.

PROFESSIONAL CONTEXT:

In production fine-tuning work (e.g. legal document summarization at AIM),
this challenge was addressed using a full LLaMA model (7B+ parameters) and
a substantially larger curated training dataset — both factors this PoC
deliberately scaled down to fit free Colab GPU constraints and a fast
demo timeline.

This PoC's value lies in demonstrating the COMPLETE fine-tuning pipeline —
data preparation, LoRA configuration, training, evaluation, and systematic
debugging — rather than claiming production-quality output from a
deliberately minimal setup.
""")
print("="*70)

In [ ]:
# ── CELL 12: Download results ──────────────────────────────────────────────────
from google.colab import files
import os

if os.path.exists('llama_finetune_results.png'):
    files.download('llama_finetune_results.png')
    print('⬇️  Downloaded: llama_finetune_results.png')